# Chapter 16: Embedding For Time Series

### Learned Emeddings

In the previous chapters, we have discussed a TimeVAE model that generates synthetic time series
data. To dive deeper into how the model works and understand time series embeddings in deep neural
networks, we will visualize the learned latent space embedding of input data in the TimeVAE model.

Recall that a TimeVAE model generates the mean and variance of the latent space distribution. We
pull this generated mean and reduce the dimensionality to 2D using dimension reduction methods
such as PCA for visualization. For easier understanding, we use the naive sine wave dataset as the
input to the model.

First, we create a embedding_extractor function to extract the latent space embeddings from a trained
TimeVAE model.

In [ ]:
from torch.utils.data import DataLoader
import lightning as L
import torch
def embedding_extractor(
 forecaster: L.LightningModule,
 x: torch.Tensor
) -> tuple[torch.Tensor]:
    """compute the embeddings based on the input
    :param forecaster: the trained forecaster
    :param x: input historical time series,
    """
    forecaster.model.to(x.device)
    z_mean, z_log_var, z = forecaster.model.encoder(
        x.type_as(forecaster.model.encoder.z_mean_layer.weight)
    )

    return z_mean, z_log_var, z

Following Chapter 12, our trained model is vae_model. For better visualizations, we recreate a batch
of the input data, input_example, with batch size 400 and time series length 24. A larger batch size
is chosen to ensure we have enough data points to visualize. We can extract the latent space embeddings as follows:

In [ ]:
vae_model = 'placeholder for vae model'
 #sine wave input
input_example = torch.sin(torch.linspace(0, 2 * torch.pi, 24 * 400))

z_mean_example, _, _ = embedding_extractor(
 vae_model,
 input_example
)

z_mean_example is our clean version of the embedding. It has dimension (400,1,8). To visualize the
input data and the embedding, we reduce the dimension to 2D using PCA:

In [ ]:
from torchdr import TSNE

n_components = 2

dr_input_result = TSNE(
 perplexity=30,
 n_components=n_components
).fit_transform(input_example.squeeze())

dr_z_result = TSNE(
 perplexity=30,
 n_components=n_components
).fit_transform(z_mean_example.squeeze())

The left panel shows a perfect circle, which is expected as the input is a sine wave. The right panel
shows the latent space embedding. It maintains the local structure of the input data, but with 1/3 of
the original size.
In real-world datasets, the learned embeddings can be more complex. However, the idea is similar.
The learned embeddings contain key patterns of the original data that are related to the task at hand.
To make embeddings more useful, some models seek more universal embeddings of time series that
can be used for different tasks.